In [1]:
import numpy as np

In [2]:
sample = np.load('/home/datasets/FLARE-MedFM/FLARE-Task1-PancancerRECIST-to-3D/train_npz/CT_Lesion_000376_23_01_033-045.npz')
sample

NpzFile '/home/datasets/FLARE-MedFM/FLARE-Task1-PancancerRECIST-to-3D/train_npz/CT_Lesion_000376_23_01_033-045.npz' with keys: imgs, gts, spacing, direction, origin

In [3]:
sample['spacing']

array([0.9375, 0.9375, 5.    ])

In [4]:
"""
Simulate RECIST 2D markers from lesion 3D mask
"""
import cv2
import numpy as np
import SimpleITK as sitk
from scipy.spatial.distance import pdist, squareform

input_npz_path = '/home/datasets/FLARE-MedFM/FLARE-Task1-PancancerRECIST-to-3D/train_npz/CT_Lesion_Chest_LIDC-IDRI-0022.npz'

input_npz_data = np.load(input_npz_path, allow_pickle=True)
print(input_npz_data.keys()) # imgs, gts, spacing, direction, origin
gt_array = input_npz_data["gts"]
lesion_ids = np.unique(gt_array)[1:]
RECIST_array = np.zeros_like(gt_array, dtype=np.uint8)


for lesion_id in lesion_ids:
    lesion_size = np.sum(gt_array == lesion_id)
    if lesion_size <= 1000: # ignore non-measure lesions (rough threshold)
        print('Non-measurable lesion with size:', lesion_size)
        continue
    else:
        # get largest 2D slice for the lesion
        lesion_array = np.uint8(gt_array == lesion_id)
        area_per_slice = np.sum(lesion_array, axis=(1, 2))
        key_slice_id = np.argmax(area_per_slice)
        largest_2D_slice = lesion_array[key_slice_id, :, :] # key slice to derive the RECIST marker
        # get points of the diameter
        points = np.column_stack(np.where(largest_2D_slice))
        dist_matrix = squareform(pdist(points))
        max_diam_idx = np.unravel_index(np.argmax(dist_matrix), dist_matrix.shape)
        p1 = points[max_diam_idx[0]]
        p2 = points[max_diam_idx[1]]
        # generate 2D link marker
        cv2.line(img=RECIST_array[key_slice_id, :, :], pt1=np.flip(p1), pt2=np.flip(p2), color=int(lesion_id), thickness=2)

# save the RECIST marker to nifti for visualization
prompt_array_sitk = sitk.GetImageFromArray(RECIST_array)
prompt_array_sitk.SetOrigin(input_npz_data["origin"])
prompt_array_sitk.SetSpacing(input_npz_data["spacing"])
prompt_array_sitk.SetDirection(input_npz_data["direction"])
sitk.WriteImage(prompt_array_sitk, "RECIST_marker_demo.nii.gz")
# save target
img_sitk = sitk.GetImageFromArray(input_npz_data["gts"])
img_sitk.CopyInformation(prompt_array_sitk)
sitk.WriteImage(img_sitk, "gt_demo.nii.gz")
# save image
img_sitk = sitk.GetImageFromArray(input_npz_data["imgs"])
img_sitk.CopyInformation(prompt_array_sitk)
sitk.WriteImage(img_sitk, "img_demo.nii.gz")


KeysView(NpzFile '/home/datasets/FLARE-MedFM/FLARE-Task1-PancancerRECIST-to-3D/train_npz/CT_Lesion_Chest_LIDC-IDRI-0022.npz' with keys: imgs, gts, spacing, direction, origin)
